#Business Transformation based on Modeling

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY cc.customer_id) AS customer_key,
    cc.customer_id,
    cc.customer_number,
    cc.first_name,
    cc.last_name,
    le.country,
    cc.marital_status,
    CASE
        WHEN cc.gender <> 'n/a' THEN cc.gender
        ELSE COALESCE(ce.gender, 'n/a')
    END AS gender,
    ce.birth_date AS birthdate,
    cc.created_date AS create_date
FROM silver.crm_customer cc
LEFT JOIN silver.erp_customer ce
    ON cc.customer_number = ce.customer_number
LEFT JOIN silver.erp_customer_location le
    ON cc.customer_number = le.customer_number
"""
df = spark.sql(query)

## Sanity Checks Dataframe

In [0]:
df.limit(10).display()

# Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("workspace.gold.dim_customer")

## Sanity Checks Gold Table

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customer LIMIT 10